<a href="https://www.kaggle.com/code/baseto/training-and-experiments?scriptVersionId=263166931" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# SHIT'S GETTING REAL, BUT NO WAM. CREATING A DATA SPLITTING FN
Here, we'll create a Python function that takes a pandas DataFrame, the number of time steps for time series prediction, and a validation/test split ratio as input. The function should group the data by country ('iso3'), create time series sequences of the specified number of steps for each country, and then split the data for each country chronologically into training, validation, and test sets based on the provided ratio. Finally, the function should combine the sets from all countries and return the concatenated training, validation, and test sets (X_train, y_train, X_val, y_val, X_test, y_test). Include error handling for countries with insufficient data.

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/agricultural-hotspot-fapar-rasters-conflict-info/processed_df.csv')

In [ ]:
df.head()

In [ ]:
df['processed_image_path'][0]

In [ ]:
# Define the old and new paths
old_path = '/content/drive/MyDrive/Colab Notebooks/Unilag project/'
new_path = '/kaggle/input/agricultural-hotspot-fapar-rasters-conflict-info/'

# Use .str.replace() to update the column
df['processed_image_path'] = df['processed_image_path'].str.replace(old_path, new_path)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.to_csv('/kaggle/working/processed_df.csv', index = False)

In [ ]:
def create_time_series_splits(df, n_steps, val_test_split_ratio):
    """
    Creates time series sequences and splits data into training, validation, and test sets
    per country.

    Args:
        df (pd.DataFrame): The input DataFrame with time series data and 'iso3' column.
        n_steps (int): The number of time steps in each input sequence (X).
        val_test_split_ratio (float): The combined proportion of data for validation and testing.

    Returns:
        tuple: A tuple containing X_train, y_train, X_val, y_val, X_test, y_test.
               Returns (None, None, None, None, None, None) if processing fails or no data.
    """
    all_X_train, all_y_train = [], []
    all_X_val, all_y_val = [], []
    all_X_test, all_y_test = [], []

    # Group by country
    grouped_data = df.groupby('iso3')

    for iso3, country_df in grouped_data:
        # Sort data chronologically
        country_df = country_df.sort_values(by=['year', 'month'])
        numeric_cols = country_df.select_dtypes(include=np.number).columns.tolist()
        country_data = country_df[numeric_cols].values

        # Check if there's enough data to create at least one sequence for train, val, and test
        required_length = n_steps + 1  # n_steps for X, 1 for y
        if len(country_data) < required_length + int(len(country_data) * val_test_split_ratio):
            print(f"Skipping country {iso3}: Insufficient data ({len(country_data)} rows) to create sequences and splits with n_steps={n_steps} and val_test_split_ratio={val_test_split_ratio}.")
            continue

        # Create time series sequences
        X, y = [], []
        for i in range(len(country_data) - n_steps):
            # Create input sequence (X)
            seq_x = country_data[i:(i + n_steps)]
            X.append(seq_x)
            seq_y = country_data[i + n_steps]
            y.append(seq_y)

        X = np.array(X)
        y = np.array(y)

        # Determine split points
        num_sequences = len(X)
        if num_sequences == 0:
             print(f"Skipping country {iso3}: No sequences could be created with n_steps={n_steps}.")
             continue

        val_test_size = int(num_sequences * val_test_split_ratio)
        test_size = val_test_size // 2
        val_size = val_test_size - test_size
        train_size = num_sequences - val_test_size

        if train_size <= 0:
             print(f"Skipping country {iso3}: Not enough sequences ({num_sequences}) for training with val_test_split_ratio={val_test_split_ratio}.")
             continue


        # Split data chronologically
        X_train = X[:train_size]
        y_train = y[:train_size]
        X_val = X[train_size:(train_size + val_size)]
        y_val = y[train_size:(train_size + val_size)]
        X_test = X[(train_size + val_size):]
        y_test = y[(train_size + val_size):]

        all_X_train.append(X_train)
        all_y_train.append(y_train)
        all_X_val.append(X_val)
        all_y_val.append(y_val)
        all_X_test.append(X_test)
        all_y_test.append(y_test)

    # Concatenate data from all countries
    if not all_X_train: # Check if any training data was generated
        print("No training data generated for any country.")
        return None, None, None, None, None, None

    X_train_combined = np.concatenate(all_X_train, axis=0)
    y_train_combined = np.concatenate(all_y_train, axis=0)
    X_val_combined = np.concatenate(all_X_val, axis=0) if all_X_val else np.array([])
    y_val_combined = np.concatenate(all_y_val, axis=0) if all_y_val else np.array([])
    X_test_combined = np.concatenate(all_X_test, axis=0) if all_X_test else np.array([])
    y_test_combined = np.concatenate(all_y_test, axis=0) if all_y_test else np.array([])


    return X_train_combined, y_train_combined, X_val_combined, y_val_combined, X_test_combined, y_test_combined


In [ ]:
import pandas as pd

def split_dataframe_chronologically_by_country(df, val_test_split_ratio):
    """
    Splits a DataFrame into training, validation, and test sets chronologically
    per country.

    Args:
        df (pd.DataFrame): The input DataFrame with time series data and 'iso3' column.
        val_test_split_ratio (float): The combined proportion of data for validation and testing.

    Returns:
        tuple: A tuple containing train_df, val_df, test_df.
               Returns (pd.DataFrame(), pd.DataFrame(), pd.DataFrame()) if no data.
    """
    all_train_dfs, all_val_dfs, all_test_dfs = [], [], []

    # Group by country
    grouped_data = df.groupby('iso3')

    for iso3, country_df in grouped_data:
        # Sort data chronologically
        country_df = country_df.sort_values(by=['year', 'month']).reset_index(drop=True)

        num_rows = len(country_df)

        # Determine split points
        val_test_size = int(num_rows * val_test_split_ratio)
        test_size = val_test_size // 2
        val_size = val_test_size - test_size
        train_size = num_rows - val_test_size

        if train_size <= 0:
             print(f"Skipping country {iso3}: Not enough data ({num_rows} rows) for training after split with val_test_split_ratio={val_test_split_ratio}.")
             continue

        # Split data chronologically
        train_df_country = country_df.iloc[:train_size].copy()
        val_df_country = country_df.iloc[train_size:(train_size + val_size)].copy()
        test_df_country = country_df.iloc[(train_size + val_size):].copy()

        all_train_dfs.append(train_df_country)
        all_val_dfs.append(val_df_country)
        all_test_dfs.append(test_df_country)

    # Concatenate dataframes from all countries
    train_df_combined = pd.concat(all_train_dfs, ignore_index=True) if all_train_dfs else pd.DataFrame()
    val_df_combined = pd.concat(all_val_dfs, ignore_index=True) if all_val_dfs else pd.DataFrame()
    test_df_combined = pd.concat(all_test_dfs, ignore_index=True) if all_test_dfs else pd.DataFrame()


    return train_df_combined, val_df_combined, test_df_combined

### Splitting DataFrames and Creating Datasets and DataLoaders

In [ ]:
# Define split ratio and parameters for Dataset and DataLoader
val_test_split_ratio = 0.3
n_steps_for_dataloader = 6
forecast_horizon = 3
output_dir = "/content/drive/MyDrive/Colab Notebooks/Unilag project/fapar_patches_processed"
batch_size = 32
num_workers = 2
target_col = 'hs_code' # Assuming the target column is 'hs_code'

# Split the DataFrame chronologically by country
train_df, val_df, test_df = split_dataframe_chronologically_by_country(df, val_test_split_ratio)

In [ ]:
train_df

In [ ]:
val_df

In [ ]:
test_df

In [ ]:
print(f"Train DataFrame shape: {train_df.shape}")
print(f"Validation DataFrame shape: {val_df.shape}")
print(f"Test DataFrame shape: {test_df.shape}")

## NOW, OFF TO *DATALOADER*
Create a custom dataset and dataloader to load processed TIFF files and corresponding numerical data from a pandas DataFrame for time series prediction, ensuring the data is in a suitable format for a machine learning model.

First. we create a Python class that inherits from a data loading library's Dataset class (`torch.utils.data.Dataset`).


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import tifffile
import os
import cv2 # Using OpenCV for resizing and padding

class ConflictDataset(Dataset):
    """Custom Dataset for loading processed FAPAR images and numerical data for multi-step forecasting."""

    def __init__(self, dataframe, img_dir, n_steps, forecast_horizon, target_col='hs_code', target_size=(256, 256)):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing numerical features and image paths.
            img_dir (str): Directory with processed image files.
            n_steps (int): Number of time steps in the input sequence (X).
            forecast_horizon (int): The number of future time steps to forecast.
            target_col (str): The name of the target column in the DataFrame. Defaults to 'hs_code'.
            target_size (tuple): The target spatial size (height, width) for image resizing/padding. Defaults to (256, 256).
        """
        self.dataframe = dataframe.copy()
        self.img_dir = img_dir
        self.n_steps = n_steps
        self.forecast_horizon = forecast_horizon
        self.target_col = target_col
        self.target_size = target_size # (height, width)

        # Pre-process the DataFrame to create sequences and store image paths and multi-step targets
        self.sequences = []
        self.image_paths = []
        self.targets = []
        self.indices = [] # Store original DataFrame index for potential debugging or joining later

        # Ensure DataFrame is sorted by country, year, and month
        self.dataframe = self.dataframe.sort_values(by=['iso3', 'year', 'month']).reset_index(drop=True)

        # Group by country
        grouped_data = self.dataframe.groupby('iso3')

        for iso3, country_df in grouped_data:
            # Exclude columns not needed for numerical features in the sequence
            # Keep 'processed_image_path' aside for loading images
            # Keep 'target_col' aside for the target variable
            feature_cols = country_df.select_dtypes(include=np.number).columns.tolist()
            if self.target_col in feature_cols:
                feature_cols.remove(self.target_col)
            if 'year' in feature_cols:
                feature_cols.remove('year')

            country_features = country_df[feature_cols].values
            country_targets_data = country_df[self.target_col].values
            country_image_paths = country_df['processed_image_path'].values
            country_indices = country_df.index.values # Original indices in the sorted df


            # Create time series sequences with multi-step targets
            # We need enough data for the input sequence (n_steps) PLUS the forecast horizon
            for i in range(len(country_df) - self.n_steps - self.forecast_horizon + 1):
                # Numerical sequence (X)
                seq_x = country_features[i:(i + self.n_steps)]
                self.sequences.append(seq_x)

                # Image paths for the sequence (X) - we need n_steps images
                seq_img_paths = country_image_paths[i:(i + self.n_steps)]
                self.image_paths.append(seq_img_paths)

                # Multi-step targets (y) - the target values for the forecast horizon steps
                # Starting from the step immediately following the input sequence
                seq_y = country_targets_data[(i + self.n_steps):(i + self.n_steps + self.forecast_horizon)]
                self.targets.append(seq_y)

                # Store the index of the last element in the input sequence (or the first target element)
                # Let's store the index of the first target element in the original sorted df
                self.indices.append(country_indices[i + self.n_steps])


        self.sequences = np.array(self.sequences)
        # image_paths are already a list of numpy arrays/lists
        self.targets = np.array(self.targets)
        self.indices = np.array(self.indices)


    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.sequences)

    def __getitem__(self, idx):
        """Retrieves a sample (sequence of features, sequence of images, multi-step target) at the given index."""
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # Get the numerical sequence and target
        numerical_sequence = self.sequences[idx]
        target = self.targets[idx]
        original_index = self.indices[idx]


        # Load and process the corresponding image sequence
        # Each item in self.image_paths[idx] is a filename relative to self.img_dir
        processed_image_sequence = []
        for img_filename in self.image_paths[idx]:
            img_path = os.path.join(self.img_dir, img_filename)
            try:
                # Load image using tifffile
                img = tifffile.imread(img_path)
                img = np.nan_to_num(img, nan=0)

                # Standardize image size
                original_height, original_width = img.shape[:2]
                target_height, target_width = self.target_size

                if original_height == target_height and original_width == target_width:
                    processed_img = img
                elif original_height > target_height or original_width > target_width:
                    processed_img = cv2.resize(img, (target_width, target_height), interpolation=cv2.INTER_AREA)
                else:
                    # Image is smaller, zero-pad it
                    # Calculate padding amounts
                    pad_height = target_height - original_height
                    pad_width = target_width - original_width

                    # Determine padding for top/bottom and left/right
                    pad_top = pad_height // 2
                    pad_bottom = pad_height - pad_top
                    pad_left = pad_width // 2
                    pad_right = pad_width - pad_left

                    # Add padding. If image has channels, pad along spatial dimensions only.
                    if img.ndim == 2: # Grayscale
                         processed_img = np.pad(img, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='constant', constant_values=0)
                    elif img.ndim == 3: # Color/Multi-channel
                         processed_img = np.pad(img, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode='constant', constant_values=0)
                    else:
                        print(f"Warning: Unexpected image dimensions for {img_filename}: {img.ndim}. Skipping padding.")
                        processed_img = img # Keep original size if dimensions are unexpected


                processed_image_sequence.append(processed_img)

            except FileNotFoundError:
                print(f"Warning: Image file not found at {img_path}. Returning None for this image.")
                processed_image_sequence.append(None) # Or handle as appropriate, e.g., a zero array
            except Exception as e:
                print(f"Error loading or processing image {img_path}: {e}. Returning None for this image.")
                processed_image_sequence.append(None)

        # Stack processed images if they are not None
        if all(img is not None for img in processed_image_sequence):
            # Ensure all images in the sequence have the same dimensions before stacking
            if not all(img.shape == processed_image_sequence[0].shape for img in processed_image_sequence):
                 print(f"Error: Inconsistent image shapes in sequence {idx}. Cannot stack.")
                 return None # Or handle error appropriately

            # Stack images along the time step dimension (axis=0)
            # Resulting shape will be (n_steps, height, width, channels)
            image_sequence_stacked = np.stack(processed_image_sequence, axis=0)
        else:
            # Handle cases where one or more images failed to load/process
            print(f"Warning: Skipping sequence {idx} due to missing/failed image loading/processing.")
            return None, None, None, None

        numerical_sequence_tensor = torch.tensor(numerical_sequence, dtype=torch.float32)

        # Ensure image_sequence_stacked has a channel dimension even if it's grayscale
        if image_sequence_stacked.ndim == 3: # (n_steps, height, width) for grayscale
             image_sequence_stacked = np.expand_dims(image_sequence_stacked, axis=-1) # Add channel dimension at the end
             # image_sequence_stacked = np.repeat(image_sequence_stacked, 3, axis=-1)


        image_sequence_tensor = torch.tensor(image_sequence_stacked, dtype=torch.float32)

        # The shape of target is (forecast_horizon,)
        target_tensor = torch.tensor(target, dtype=torch.long)

        return numerical_sequence_tensor, image_sequence_tensor, target_tensor, original_index

### Create a dataloader
Create an instance of a DataLoader (e.g., `torch.utils.data.DataLoader`) using your custom Dataset. This will handle batching, shuffling, and parallel loading of data.


In [ ]:
# Define split ratio and parameters for Dataset and DataLoader
val_test_split_ratio = 0.3
n_steps_for_dataloader = 6
forecast_horizon = 3
output_dir = "/kaggle/input/agricultural-hotspot-fapar-rasters-conflict-info/fapar_patches_processed"
batch_size = 4
num_workers = 2
target_col = 'hs_code' # Assuming the target column is 'hs_code'

# BUILDING MODELS

## CNN MODEL
we create a basic model using a 2D CNN to extract features from image sequences and combine them with numerical features before passing them to a time series network. The model should take the number of features to extract from the images as a variable. The output of the CNN feature extractor should have the shape `[BATCH, num_steps, num_features]`.

In [ ]:
import torch.nn as nn
import torch # Import torch for AdaptiveAvgPool3d

class ImageSequenceFeatureExtractor3D(nn.Module):
    """
    3D CNN feature extractor for image sequences.
    Processes the image sequence with 3D convolutions and pooling,
    with optional output modes for feature combination or direct classification.
    """
    def __init__(self, num_time_steps, in_channels=3, num_features_extracted=None,
                 combining_outputs=True, forecast_horizon=1, num_classes=4):
        """
        Args:
            num_time_steps (int): The number of time steps in the input sequence.
            in_channels (int): The number of input channels in each image (e.g., 3 for RGB).
            num_features_extracted (int, optional): The number of features to extract per time step
                when combining_outputs is True. Required if combining_outputs is True.
            combining_outputs (bool): If True, the output is a tensor of shape
                [batch_size, num_steps, num_features_extracted] for feature combination.
                If False, the output is a tensor of shape [batch_size, forecast_horizon, num_classes]
                for a multi-step classification task. Defaults to True.
            forecast_horizon (int): The number of time steps to forecast. Used when
                combining_outputs is False. Defaults to 1.
            num_classes (int): The number of classes for each classification output.
                Used when combining_outputs is False. Defaults to 4.
        """
        super(ImageSequenceFeatureExtractor3D, self).__init__()

        self.combining_outputs = combining_outputs
        self.forecast_horizon = forecast_horizon
        self.num_classes = num_classes
        self.num_time_steps = num_time_steps # Keep track of num_time_steps

        # The input shape to Conv3d is (batch_size, channels, depth, height, width)
        # Our image sequence is (batch_size, num_steps, height, width, channels) initially from DataLoader
        # We need to permute it to (batch_size, channels, num_steps, height, width) for Conv3d

        self.features = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)), # Pool spatially, keep time dimension

            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)), # Pool spatially, keep time dimension

            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)), # Pool spatially, keep time dimension
        )

        # Add the AdaptiveAvgPool3d layer
        # It will pool over the last two dimensions (height and width) to output size (1, 1)
        # The output shape after this will be (batch_size, 128, num_steps, 1, 1)
        self.avgpool = nn.AdaptiveAvgPool3d((None, 1, 1)) # Pool over spatial dimensions to 1x1, keep time dimension

        # After adaptive pooling, shape will be (batch_size, 128, num_steps, 1, 1)
        # We can then flatten the spatial dimensions (1, 1)
        # Output shape: (batch_size, 128, num_steps)

        # Define the final layer(s) based on combining_outputs
        flattened_size_per_time_step = 128

        if self.combining_outputs:
            if num_features_extracted is None:
                 raise ValueError("num_features_extracted must be specified when combining_outputs is True")
            self.output_layer = nn.Conv1d(flattened_size_per_time_step, num_features_extracted, kernel_size=1)
            self.num_features_extracted = num_features_extracted # Store for clarity
        else:
             # The output will be reshaped to (batch_size, forecast_horizon, num_classes)
             # We need to process the sequence of features (shape [batch_size, 128, num_steps])
             # to output forecast_horizon * num_classes values per batch item.
             # This requires a layer that can handle the time dimension.
             # A simple approach is to flatten the features across time steps and apply a linear layer,
             # but this loses temporal information.
             # A better approach might be to use a time series layer (like LSTM) or
             # process the sequence with 1D convolutions.
             # Given the structure of ImageSequenceFeatureExtractor2D (which produced a single
             # vector per sequence for the combining_outputs=False case), let's adapt that logic.
             # Flatten the features across time steps and channels: [batch_size, 128 * num_steps]
             # Then apply a linear layer to output forecast_horizon * num_classes.

             # This requires knowing num_steps at init time to calculate the flattened size.
             # So, the `num_time_steps` parameter is crucial here.
             flattened_size_across_time = flattened_size_per_time_step * num_time_steps
             self.output_layer = nn.Linear(flattened_size_across_time, forecast_horizon * num_classes)


    def forward(self, x):
        """
        Forward pass of the 3D CNN feature extractor.

        Args:
            x (torch.Tensor): Input image sequence tensor
                              Expected shape: [batch_size, num_steps, height, width, channels]

        Returns:
            torch.Tensor: Extracted features. The shape depends on the combining_outputs parameter:
                          - If combining_outputs=True: [batch_size, num_steps, num_features_extracted]
                          - If combining_outputs=False: [batch_size, forecast_horizon, num_classes]
        """
        batch_size, num_steps, height, width, channels = x.size()

        # Permute the input to match Conv3d input shape (batch_size, channels, depth, height, width)
        # Depth is the time dimension here
        x = x.permute(0, 4, 1, 2, 3) # From [batch, steps, H, W, C] to [batch, C, steps, H, W]

        # Pass through convolutional and pooling layers
        # Output shape: [batch_size, 128, num_steps, H', W'] where H', W' are reduced spatial dimensions
        x = self.features(x)

        # Apply adaptive spatial pooling
        # Output shape: [batch_size, 128, num_steps, 1, 1]
        x = self.avgpool(x)

        # Remove the spatial dimensions (1, 1)
        # Output shape: [batch_size, 128, num_steps]
        x = x.squeeze(4).squeeze(3)

        if self.combining_outputs:
            # Apply the Conv1d layer to get features per time step
            # Input shape to Conv1d is (batch_size, in_channels, length) -> (batch_size, 128, num_steps)
            # Output shape is (batch_size, num_features_extracted, num_steps)
            x = self.output_layer(x)

            # Permute back to [batch_size, num_steps, num_features_extracted]
            x = x.permute(0, 2, 1)
        else:
            # Flatten the features across time steps and channels
            # Output shape: [batch_size, 128 * num_steps]
            x = x.view(batch_size, -1)

            # Apply the linear output layer for classification
            # Output shape: [batch_size, forecast_horizon * num_classes]
            x = self.output_layer(x)

            # Reshape to [batch_size, forecast_horizon, num_classes]
            x = x.view(batch_size, self.forecast_horizon, self.num_classes)

        return x

In [ ]:
import torch.nn as nn
import torch

class ImageSequenceFeatureExtractor2D(nn.Module):
    """
    2D CNN feature extractor for image sequences.
    Reshapes the input from (batch, num_steps, channels, H, W)
    to (batch, num_steps * channels, H, W) before processing.
    """
    def __init__(self, num_time_steps, in_channels=3, num_features_extracted=None,
                 combining_outputs=False, forecast_horizon=1, num_classes=4):
        """
        Args:
            num_time_steps (int): The number of time steps in the input sequence.
            in_channels (int): The number of input channels per time step (e.g., 3 for RGB).
            num_features_extracted (int, optional): The number of features to extract
                when combining_outputs is True.
            combining_outputs (bool): If True, the output is a single feature vector
                of size num_features_extracted. If False, the output is a tensor
                for a classification task over a forecast horizon.
            forecast_horizon (int): The number of time steps to forecast. Used when
                combining_outputs is False.
            num_classes (int): The number of classes for each classification output.
                Used when combining_outputs is False.
        """
        super(ImageSequenceFeatureExtractor2D, self).__init__()

        # Store parameters for use in the forward pass
        self.combining_outputs = combining_outputs
        self.forecast_horizon = forecast_horizon
        self.num_classes = num_classes
        self.num_time_steps = num_time_steps
        self.num_features_extracted = num_features_extracted

        # The effective number of input channels for the 2D CNN is the product of
        # the original channels and the number of time steps.
        in_channels_2d = in_channels * num_time_steps

        # Define the 2D CNN feature extraction layers
        self.features = nn.Sequential(
            # Input shape: (batch_size, in_channels_2d, height, width)
            nn.Conv2d(in_channels_2d, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Pool spatially

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Pool spatially

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Pool spatially
        )

        # We use AdaptiveAvgPool2d to pool over the entire spatial dimensions
        # to get a single feature vector per sequence.
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # The final layer depends on the combining_outputs parameter
        flattened_size_after_pool = 128
        if self.combining_outputs:
            if num_features_extracted is None:
                raise ValueError("num_features_extracted must be specified when combining_outputs is True")
            self.output_layer = nn.Linear(flattened_size_after_pool, num_time_steps * num_features_extracted)
        else:
            # The output will be reshaped to (batch_size, forecast_horizon, num_classes)
            self.output_layer = nn.Linear(flattened_size_after_pool, forecast_horizon * num_classes)

    def forward(self, x):
        """
        Forward pass of the 2D CNN feature extractor.

        Args:
            x (torch.Tensor): Input image sequence tensor.
                              Expected shape: [batch_size, num_steps, channels, height, width]

        Returns:
            torch.Tensor: Extracted features per sequence. The shape depends on the
                          combining_outputs parameter.
        """
        # Reshape the input tensor to combine the time steps and channels
        # From [batch, steps, channels, H, W] to [batch, steps*channels, H, W]
        batch_size = x.size(0)
        x = x.view(batch_size, -1, x.size(2), x.size(3))

        # Pass the reshaped tensor through the 2D CNN layers
        x = self.features(x)

        # Apply adaptive spatial pooling
        # Output shape: (batch_size, 128, 1, 1)
        x = self.avgpool(x)

        # Flatten the output for the final layer
        # Output shape: (batch_size, 128)
        x = torch.flatten(x, 1)

        # Pass through the final output layer
        x = self.output_layer(x)

        # If we are not combining outputs, reshape to the forecast horizon and num classes
        if self.combining_outputs:
            # Output shape: (batch_size, num_time_steps * num_features_extracted)
            x = x.view(batch_size, self.num_time_steps, self.num_features_extracted)
        else:
            # Output shape: (batch_size, forecast_horizon, num_classes)
            x = x.view(batch_size, self.forecast_horizon, self.num_classes)

        return x

### TESTING THE CNN NETWORK

#### Testing ImageSequenceFeatureExtractor2D for both cases

In [ ]:
# Test the ImageSequenceFeatureExtractor2D for both combining_outputs cases
import torch

# Define test parameters
test_batch_size = 4
test_num_time_steps = 5 # Corresponds to n_steps
test_height = 256
test_width = 256
test_channels = 3 # Assuming 3 channels for the image input
test_num_features_extracted = 64 # For the case when combining_outputs is True
test_forecast_horizon = 3 # For the case when combining_outputs is False
test_num_classes = 4 # For the case when combining_outputs is False


# Generate a random tensor with the specified shape
# The shape should be [batch_size, num_steps, height, width, channels]
dummy_image_sequences = torch.randn(test_batch_size, test_num_time_steps, test_height, test_width, test_channels)


# Instantiate ImageSequenceFeatureExtractor2D with combining_outputs=True
print("Testing with combining_outputs=True:")
image_feature_extractor_combined = ImageSequenceFeatureExtractor2D(
    num_time_steps=test_num_time_steps,
    in_channels=test_channels,
    num_features_extracted=test_num_features_extracted,
    combining_outputs=True
)

# Move the dummy tensor to the same device as the model (if using GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_feature_extractor_combined.to(device)
dummy_image_sequences = dummy_image_sequences.to(device)

# Pass the dummy tensor through the model
with torch.no_grad(): # No need to calculate gradients for testing
    extracted_features_combined = image_feature_extractor_combined(dummy_image_sequences)

# Print the shape of the output tensor to verify
print(f"Input dummy image sequence shape: {dummy_image_sequences.shape}")
print(f"Extracted features shape (combining_outputs=True): {extracted_features_combined.shape}")

# Expected output shape when combining_outputs=True: [batch_size, num_steps * num_features_extracted]
# The output layer is nn.Linear(flattened_size_after_pool, num_time_steps * num_features_extracted)
expected_shape_combined = (test_batch_size, test_num_time_steps, test_num_features_extracted)


if extracted_features_combined.shape == expected_shape_combined:
    print("Output shape matches the expected shape for combining_outputs=True.")
else:
    print(f"Output shape {extracted_features_combined.shape} does not match the expected shape {expected_shape_combined} for combining_outputs=True.")

print("-" * 30)

# Instantiate ImageSequenceFeatureExtractor2D with combining_outputs=False
print("Testing with combining_outputs=False:")
image_feature_extractor_forecast = ImageSequenceFeatureExtractor2D(
    num_time_steps=test_num_time_steps,
    in_channels=test_channels,
    combining_outputs=False,
    forecast_horizon=test_forecast_horizon,
    num_classes=test_num_classes
)

# Move the model to the same device
image_feature_extractor_forecast.to(device)

# Pass the dummy tensor through the model
with torch.no_grad():
    extracted_features_forecast = image_feature_extractor_forecast(dummy_image_sequences)

# Print the shape of the output tensor to verify
print(f"Input dummy image sequence shape: {dummy_image_sequences.shape}")
print(f"Extracted features shape (combining_outputs=False): {extracted_features_forecast.shape}")

# Expected output shape when combining_outputs=False: [batch_size, forecast_horizon, num_classes]
expected_shape_forecast = (test_batch_size, test_forecast_horizon, test_num_classes)

if extracted_features_forecast.shape == expected_shape_forecast:
    print("Output shape matches the expected shape for combining_outputs=False.")
else:
    print(f"Output shape {extracted_features_forecast.shape} does not match the expected shape {expected_shape_forecast} for combining_outputs=False.")

#### Testing ImageSequenceFeatureExtractor3D with different output modes

In [ ]:
# Test the ImageSequenceFeatureExtractor3D for both combining_outputs cases
import torch

# Define test parameters
test_batch_size = 4
test_num_time_steps = 5 # Corresponds to n_steps
test_height = 256
test_width = 256
test_channels = 3 # Assuming 3 channels for the image input
test_num_features_extracted = 64 # For the case when combining_outputs is True
test_forecast_horizon = 3 # For the case when combining_outputs is False
test_num_classes = 4 # For the case when combining_outputs is False


# Generate a random tensor with the specified shape
# The shape should be [batch_size, num_steps, height, width, channels]
dummy_image_sequences = torch.randn(test_batch_size, test_num_time_steps, test_height, test_width, test_channels)


# Instantiate ImageSequenceFeatureExtractor3D with combining_outputs=True
print("Testing ImageSequenceFeatureExtractor3D with combining_outputs=True:")
image_feature_extractor_combined_3d = ImageSequenceFeatureExtractor3D(
    num_time_steps=test_num_time_steps,
    in_channels=test_channels,
    num_features_extracted=test_num_features_extracted,
    combining_outputs=True
)

# Move the dummy tensor to the same device as the model (if using GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_feature_extractor_combined_3d.to(device)
dummy_image_sequences = dummy_image_sequences.to(device)

# Pass the dummy tensor through the model
with torch.no_grad(): # No need to calculate gradients for testing
    extracted_features_combined_3d = image_feature_extractor_combined_3d(dummy_image_sequences)

# Print the shape of the output tensor to verify
print(f"Input dummy image sequence shape: {dummy_image_sequences.shape}")
print(f"Extracted features shape (combining_outputs=True): {extracted_features_combined_3d.shape}")

# Expected output shape when combining_outputs=True: [batch_size, num_steps, num_features_extracted]
expected_shape_combined_3d = (test_batch_size, test_num_time_steps, test_num_features_extracted)


if extracted_features_combined_3d.shape == expected_shape_combined_3d:
    print("Output shape matches the expected shape for combining_outputs=True.")
else:
    print(f"Output shape {extracted_features_combined_3d.shape} does not match the expected shape {expected_shape_combined_3d} for combining_outputs=True.")

print("-" * 30)

# Instantiate ImageSequenceFeatureExtractor3D with combining_outputs=False
print("Testing ImageSequenceFeatureExtractor3D with combining_outputs=False:")
image_feature_extractor_forecast_3d = ImageSequenceFeatureExtractor3D(
    num_time_steps=test_num_time_steps,
    in_channels=test_channels,
    combining_outputs=False,
    forecast_horizon=test_forecast_horizon,
    num_classes=test_num_classes
)

# Move the model to the same device
image_feature_extractor_forecast_3d.to(device)

# Pass the dummy tensor through the model
with torch.no_grad():
    extracted_features_forecast_3d = image_feature_extractor_forecast_3d(dummy_image_sequences)

# Print the shape of the output tensor to verify
print(f"Input dummy image sequence shape: {dummy_image_sequences.shape}")
print(f"Extracted features shape (combining_outputs=False): {extracted_features_forecast_3d.shape}")

# Expected output shape when combining_outputs=False: [batch_size, forecast_horizon, num_classes]
expected_shape_forecast_3d = (test_batch_size, test_forecast_horizon, test_num_classes)

if extracted_features_forecast_3d.shape == expected_shape_forecast_3d:
    print("Output shape matches the expected shape for combining_outputs=False.")
else:
    print(f"Output shape {extracted_features_forecast_3d.shape} does not match the expected shape {expected_shape_forecast_3d} for combining_outputs=False.")

## TIME-SERIES NETWORK

In [ ]:
import torch.nn as nn
import torch

class TimeSeriesNetwork(nn.Module):
    """
    Time series network using LSTM to process combined numerical and image features,
    with optional output modes.
    """
    def __init__(self, combined_feature_size, hidden_size, num_layers,
                 output_mode='classification', forecast_horizon=1, num_classes=4, dropout_prob=0.0):
        """
        Args:
            combined_feature_size (int): The size of the input features (numerical + image features).
            hidden_size (int): The number of features in the hidden state of the LSTM.
            num_layers (int): Number of recurrent layers.
            output_mode (str): 'hidden_state' to output the final hidden state,
                               'classification' to output classification logits for the forecast horizon.
                               Defaults to 'classification'.
            forecast_horizon (int): The number of future time steps to forecast. Used when
                                    output_mode is 'classification'. Defaults to 1.
            num_classes (int): The number of classes for classification. Used when
                               output_mode is 'classification'. Defaults to 4.
            dropout_prob (float): Dropout probability for the LSTM layers (except the last one).
        """
        super(TimeSeriesNetwork, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_mode = output_mode
        self.forecast_horizon = forecast_horizon
        self.num_classes = num_classes


        # Define the LSTM layer
        # batch_first=True means input and output tensors are provided as (batch, seq, feature)
        self.lstm = nn.LSTM(combined_feature_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)

        # Define the output layer based on the output mode
        if self.output_mode == 'classification':
            # Linear layer to output forecast_horizon * num_classes values
            self.output_layer = nn.Linear(hidden_size, self.forecast_horizon * self.num_classes)
        elif self.output_mode == 'hidden_state':
            # If outputting hidden state, no additional linear layer needed for the main output path
            # The output will be the last hidden state (hn[-1])
            self.output_layer = nn.Identity() # Placeholder, no operation needed
        else:
            raise ValueError(f"Invalid output_mode: {output_mode}. Must be 'hidden_state' or 'classification'.")


    def forward(self, combined_features):
        """
        Forward pass of the time series network.

        Args:
            combined_features (torch.Tensor): Input tensor containing combined
                                              numerical and image features.
                                              Expected shape: [batch_size, num_steps, combined_feature_size]

        Returns:
            torch.Tensor: Processed features from the time series network.
                          If output_mode is 'hidden_state', shape is [batch_size, hidden_size].
                          If output_mode is 'classification', shape is [batch_size, forecast_horizon, num_classes].
        """
        # Initialize hidden and cell states
        # These should have shape (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, combined_features.size(0), self.hidden_size).to(combined_features.device)
        c0 = torch.zeros(self.num_layers, combined_features.size(0), self.hidden_size).to(combined_features.device)


        # Pass input through the LSTM layer
        # output shape: (batch_size, seq_len, hidden_size * num_directions)
        # (hn, cn) are the hidden state and cell state at the last time step for each layer
        output, (hn, cn) = self.lstm(combined_features, (h0, c0))

        # Apply dropout to the LSTM output
        output = self.dropout(output)

        # Determine the output based on the output mode
        if self.output_mode == 'hidden_state':
            # Return the hidden state of the last layer at the last time step
            # hn shape: (num_layers, batch_size, hidden_size)
            final_output = hn[-1] # Hidden state of the last layer
        elif self.output_mode == 'classification':
            # Use the output from the last time step for classification
            # output shape: (batch_size, seq_len, hidden_size)
            last_time_step_output = output[:, -1, :]
            # Pass the last time step output through the classification layer
            # Output shape: [batch_size, forecast_horizon * num_classes]
            raw_predictions = self.output_layer(last_time_step_output)
            # Reshape to [batch_size, forecast_horizon, num_classes]
            final_output = raw_predictions.view(-1, self.forecast_horizon, self.num_classes)


        return final_output

### TESTING THE TIME SERIES NETWORK

In [ ]:
# Test the TimeSeriesNetwork
import torch

# Define test parameters
test_batch_size = 4
test_num_time_steps = 5 # Corresponds to n_steps
test_numerical_feature_size = 10 # Dummy size for numerical features
test_image_feature_size = 64 # Dummy size for image features (should match num_features_extracted from image model)
test_combined_feature_size = test_numerical_feature_size + test_image_feature_size
test_hidden_size = 128
test_num_layers = 2
test_forecast_horizon = 3
test_num_classes = 4
test_dropout_prob = 0.0 # Use 0 dropout for simple testing

# Generate a random tensor with the expected input shape for TimeSeriesNetwork
# Shape: [batch_size, num_steps, combined_feature_size]
dummy_combined_features = torch.randn(test_batch_size, test_num_time_steps, test_combined_feature_size)

# Instantiate the TimeSeriesNetwork with output_mode='hidden_state'
print("Testing TimeSeriesNetwork with output_mode='hidden_state':")
time_series_network_hidden_state = TimeSeriesNetwork(
    combined_feature_size=test_combined_feature_size,
    hidden_size=test_hidden_size,
    num_layers=test_num_layers,
    output_mode='hidden_state',
    dropout_prob=test_dropout_prob
)

# Move the dummy tensor to the same device as the model (if using GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
time_series_network_hidden_state.to(device)
dummy_combined_features = dummy_combined_features.to(device)

# Pass the dummy tensor through the network
with torch.no_grad(): # No need to calculate gradients for testing
    ts_output_hidden_state = time_series_network_hidden_state(dummy_combined_features)

# Print the shape of the output tensor to verify
print(f"Input dummy combined features shape: {dummy_combined_features.shape}")
print(f"TimeSeriesNetwork output shape (hidden_state mode): {ts_output_hidden_state.shape}")

# Expected output shape for 'hidden_state' mode: [batch_size, hidden_size]
expected_shape_hidden_state = (test_batch_size, test_hidden_size)

if ts_output_hidden_state.shape == expected_shape_hidden_state:
    print("TimeSeriesNetwork output shape matches the expected shape for 'hidden_state' mode.")
else:
    print(f"TimeSeriesNetwork output shape {ts_output_hidden_state.shape} does not match the expected shape {expected_shape_hidden_state} for 'hidden_state' mode.")

print("-" * 30)

# Instantiate the TimeSeriesNetwork with output_mode='classification'
print("Testing TimeSeriesNetwork with output_mode='classification':")
time_series_network_classification = TimeSeriesNetwork(
    combined_feature_size=test_combined_feature_size,
    hidden_size=test_hidden_size,
    num_layers=test_num_layers,
    output_mode='classification',
    forecast_horizon=test_forecast_horizon,
    num_classes=test_num_classes,
    dropout_prob=test_dropout_prob
)

# Move the model to the same device
time_series_network_classification.to(device)

# Pass the dummy tensor through the network
with torch.no_grad():
    ts_output_classification = time_series_network_classification(dummy_combined_features)

# Print the shape of the output tensor to verify
print(f"Input dummy combined features shape: {dummy_combined_features.shape}")
print(f"TimeSeriesNetwork output shape (classification mode): {ts_output_classification.shape}")

# Expected output shape for 'classification' mode: [batch_size, forecast_horizon, num_classes]
expected_shape_classification = (test_batch_size, test_forecast_horizon, test_num_classes)

if ts_output_classification.shape == expected_shape_classification:
    print("TimeSeriesNetwork output shape matches the expected shape for 'classification' mode.")
else:
    print(f"TimeSeriesNetwork output shape {ts_output_classification.shape} does not match the expected shape {expected_shape_classification} for 'classification' mode.")

## COMBINED MODEL

In [ ]:
import torch.nn as nn
import torch

class CombinedModel(nn.Module):
    """
    Combined model that processes image sequences with a selected CNN feature extractor
    (2D or 3D) and numerical features with a time series network for multi-step
    classification forecasting.
    """
    def __init__(self, num_time_steps, image_in_channels, num_features_extracted, numerical_feature_size, hidden_size, num_layers, forecast_horizon, num_classes=2, dropout_prob=0.0, image_feature_extractor_type='3D'):
        """
        Args:
            num_time_steps (int): The number of time steps in the input sequence.
            image_in_channels (int): The number of input channels in each image (e.g., 3 for RGB).
            num_features_extracted (int): The number of features to extract per time step from the image.
            numerical_feature_size (int): The number of numerical features per time step.
            hidden_size (int): The number of features in the hidden state of the LSTM.
            num_layers (int): Number of recurrent layers in the LSTM.
            forecast_horizon (int): The number of future time steps to forecast.
            num_classes (int): The number of output classes for the final prediction layer (for classification). Defaults to 2.
            dropout_prob (float): Dropout probability. Defaults to 0.0.
            image_feature_extractor_type (str): The type of image feature extractor to use.
                                                Can be '2D' or '3D'. Defaults to '3D'.
        """
        super(CombinedModel, self).__init__()

        self.forecast_horizon = forecast_horizon
        self.num_classes = num_classes
        self.num_time_steps = num_time_steps
        self.numerical_feature_size = numerical_feature_size
        self.num_features_extracted = num_features_extracted
        self.image_feature_extractor_type = image_feature_extractor_type

        # Instantiate the selected image feature extractor
        if self.image_feature_extractor_type == '3D':
            self.image_feature_extractor = ImageSequenceFeatureExtractor3D(
                num_time_steps=num_time_steps,
                in_channels=image_in_channels,
                num_features_extracted=num_features_extracted,
                combining_outputs=True # Always True when used in CombinedModel
            )
        elif self.image_feature_extractor_type == '2D':
            self.image_feature_extractor = ImageSequenceFeatureExtractor2D(
                num_time_steps=num_time_steps,
                in_channels=image_in_channels,
                num_features_extracted=num_features_extracted,
                combining_outputs=True # Always True when used in CombinedModel
            )
        else:
            raise ValueError(f"Invalid image_feature_extractor_type: {image_feature_extractor_type}. Must be '2D' or '3D'.")


        # The size of the combined features per time step
        combined_feature_size = numerical_feature_size + num_features_extracted

        # Instantiate the time series network
        # The TimeSeriesNetwork in CombinedModel will always output the hidden state
        # to be processed by the multi-step prediction layer.
        self.time_series_network = TimeSeriesNetwork(
            combined_feature_size=combined_feature_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            output_mode='hidden_state', # Always output hidden state in CombinedModel
            dropout_prob=dropout_prob
        )

        # The TimeSeriesNetwork outputs [batch_size, hidden_size] from the last time step.
        # We need a linear layer to predict the forecast horizon steps for classification.
        self.multi_step_prediction_layer = nn.Linear(hidden_size, self.forecast_horizon * self.num_classes)


    def forward(self, numerical_sequences, image_sequences):
        """
        Forward pass of the combined model for multi-step classification forecasting.

        Args:
            numerical_sequences (torch.Tensor): Tensor containing numerical sequences.
                                                Expected shape: [batch_size, num_steps, numerical_feature_size]
            image_sequences (torch.Tensor): Tensor containing image sequences.
                                            Expected shape: [batch_size, num_steps, height, width, channels]

        Returns:
            torch.Tensor: Model predictions (logits for classification) for each step in the forecast horizon.
                          Output shape: [batch_size, forecast_horizon, num_classes]
        """
        # Process image sequences to get features per time step
        # image_features shape depends on the selected extractor but is processed to be
        # [batch_size, num_steps, num_features_extracted] by the extractors when combining_outputs=True
        image_features = self.image_feature_extractor(image_sequences)

        # Concatenate numerical and image features along the feature dimension (dim=-1)
        # Combined features shape: [batch_size, num_steps, numerical_feature_size + num_features_extracted]
        combined_features = torch.cat((numerical_sequences, image_features), dim=-1)

        # Pass combined features through the time series network
        # ts_output shape: [batch_size, hidden_size] (output from the last time step of LSTM)
        ts_output = self.time_series_network(combined_features)

        # Pass the time series output through the multi-step prediction layer
        # raw_predictions shape: [batch_size, forecast_horizon * num_classes]
        raw_predictions = self.multi_step_prediction_layer(ts_output)

        # Reshape the output to [batch_size, forecast_horizon, num_classes]
        predictions = raw_predictions.view(-1, self.forecast_horizon, self.num_classes)

        return predictions

### Testing the Combined Model

In [ ]:
# Test the CombinedModel
import torch

# Define test parameters (should align with model instantiation parameters)
test_batch_size = 4
test_num_time_steps = 5 # Corresponds to n_steps
test_height = 256
test_width = 256
test_channels = 3
test_num_features_extracted = 64 # Number of features extracted per time step from the image
test_numerical_feature_size = 13 # Should match the size after excluding 'year', 'month', 'hs_code'
test_hidden_size = 128
test_num_layers = 2
test_forecast_horizon = 3
test_num_classes = 4
test_dropout_prob = 0.2 # Use the same dropout as in the training setup

# Generate dummy input data for the CombinedModel
dummy_numerical_sequences = torch.randn(test_batch_size, test_num_time_steps, test_numerical_feature_size)
dummy_image_sequences = torch.randn(test_batch_size, test_num_time_steps, test_height, test_width, test_channels)

# Instantiate the CombinedModel using the 3D image feature extractor
print("Testing CombinedModel with ImageSequenceFeatureExtractor3D:")
combined_model_3d = CombinedModel(
    num_time_steps=test_num_time_steps,
    image_in_channels=test_channels,
    num_features_extracted=test_num_features_extracted,
    numerical_feature_size=test_numerical_feature_size,
    hidden_size=test_hidden_size,
    num_layers=test_num_layers,
    forecast_horizon=test_forecast_horizon,
    num_classes=test_num_classes,
    dropout_prob=test_dropout_prob,
    image_feature_extractor_type='3D' # Specify 3D extractor
)

# Move dummy data and model to the same device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
combined_model_3d.to(device)
dummy_numerical_sequences = dummy_numerical_sequences.to(device)
dummy_image_sequences = dummy_image_sequences.to(device)

# Pass dummy data through the combined model
with torch.no_grad():
    combined_output_3d = combined_model_3d(dummy_numerical_sequences, dummy_image_sequences)

# Print and verify the output shape
print(f"Input dummy numerical sequences shape: {dummy_numerical_sequences.shape}")
print(f"Input dummy image sequences shape: {dummy_image_sequences.shape}")
print(f"CombinedModel output shape (with 3D extractor): {combined_output_3d.shape}")

# Expected output shape for multi-step classification forecasting: [batch_size, forecast_horizon, num_classes]
expected_output_shape = (test_batch_size, test_forecast_horizon, test_num_classes)

if combined_output_3d.shape == expected_output_shape:
    print("CombinedModel output shape matches the expected shape with 3D extractor.")
else:
    print(f"CombinedModel output shape {combined_output_3d.shape} does not match the expected shape {expected_output_shape} with 3D extractor.")

print("-" * 30)

# Instantiate the CombinedModel using the 2D image feature extractor
print("Testing CombinedModel with ImageSequenceFeatureExtractor2D:")
combined_model_2d = CombinedModel(
    num_time_steps=test_num_time_steps,
    image_in_channels=test_channels,
    num_features_extracted=test_num_features_extracted,
    numerical_feature_size=test_numerical_feature_size,
    hidden_size=test_hidden_size,
    num_layers=test_num_layers,
    forecast_horizon=test_forecast_horizon,
    num_classes=test_num_classes,
    dropout_prob=test_dropout_prob,
    image_feature_extractor_type='2D' # Specify 2D extractor
)

# Move model to the same device
combined_model_2d.to(device)

# Pass dummy data through the combined model
with torch.no_grad():
    combined_output_2d = combined_model_2d(dummy_numerical_sequences, dummy_image_sequences)

# Print and verify the output shape
print(f"CombinedModel output shape (with 2D extractor): {combined_output_2d.shape}")

if combined_output_2d.shape == expected_output_shape:
    print("CombinedModel output shape matches the expected shape with 2D extractor.")
else:
    print(f"CombinedModel output shape {combined_output_2d.shape} does not match the expected shape {expected_output_shape} with 2D extractor.")

# TRAINING

## Defining Training and Evaluation Class and Functions

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import tifffile
import os
import cv2 # Using OpenCV for resizing and padding
from sklearn.preprocessing import StandardScaler
import math # Import math for CosineAnnealingWarmRestarts
import copy # Import copy for model checkpointing
import time # Import time for measuring epoch duration
from sklearn.metrics import accuracy_score # Import accuracy_score

# Assuming the ImageSequenceFeatureExtractor2D, ImageSequenceFeatureExtractor3D,
# TimeSeriesNetwork, and CombinedModel classes are defined in previous cells.
# Make sure those cells have been executed before running this class.

# Define a base Callback class
class Callback:
    def on_train_begin(self, logs=None):
        pass

    def on_epoch_begin(self, epoch, logs=None):
        pass

    def on_epoch_end(self, epoch, logs=None):
        pass

    def on_batch_begin(self, batch, logs=None):
        pass

    def on_batch_end(self, batch, logs=None):
        pass

    def on_train_end(self, logs=None):
        pass

    def set_trainer(self, trainer):
        self.trainer = trainer

    def set_model(self, model):
        self.model = model

    def set_optimizer(self, optimizer):
        self.optimizer = optimizer

    def set_dataloader(self, dataloader_name, dataloader):
        setattr(self, f"{dataloader_name}_dataloader", dataloader)


# Define specific Callback classes

class EarlyStopping(Callback):
    """Stop training when a monitored metric has stopped improving."""
    def __init__(self, monitor='val_loss', min_delta=0, patience=10, mode='auto', restore_best_weights=False):
        super().__init__()
        self.monitor = monitor
        self.min_delta = min_delta
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_weights = None
        self.wait = 0
        self.stopped_epoch = 0

        if mode not in ['auto', 'min', 'max']:
            raise ValueError("mode must be one of 'auto', 'min', 'max'")
        self.mode = mode

        if self.mode == 'min':
            self.monitor_op = np.less
            self.best = np.Inf
        elif self.mode == 'max':
            self.monitor_op = np.greater
            self.best = -np.Inf
        else: # auto
            if 'acc' in self.monitor or 'f1' in self.monitor or 'auc' in self.monitor: # Example metrics where higher is better
                self.monitor_op = np.greater
                self.best = -np.Inf
            else: # Assume loss, where lower is better
                self.monitor_op = np.less
                self.best = np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            print(f"EarlyStopping: Metric '{self.monitor}' not found in logs.")
            return

        if self.monitor_op(current, self.best - self.min_delta if self.monitor_op == np.less else self.best + self.min_delta):
            self.best = current
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = copy.deepcopy(self.trainer.model.state_dict())
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.trainer.stop_training = True # Signal the trainer to stop

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0 and self.restore_best_weights and self.best_weights is not None:
            print(f"Restoring model weights from epoch {self.stopped_epoch - self.wait}.")
            self.trainer.model.load_state_dict(self.best_weights)


class ModelCheckpoint(Callback):
    """Save the model periodically."""
    def __init__(self, filepath, monitor='val_loss', verbose=0, save_best_only=False, mode='auto', save_weights_only=False):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.save_best_only = save_best_only
        self.save_weights_only = save_weights_only

        if mode not in ['auto', 'min', 'max']:
            raise ValueError("mode must be one of 'auto', 'min', 'max'")
        self.mode = mode

        if self.mode == 'min':
            self.monitor_op = np.less
            self.best = np.Inf
        elif self.mode == 'max':
            self.monitor_op = np.greater
            self.best = -np.Inf
        else: # auto
             if 'acc' in self.monitor or 'f1' in self.monitor or 'auc' in self.monitor: # Example metrics where higher is better
                 self.monitor_op = np.greater
                 self.best = -np.Inf
             else: # Assume loss, where lower is better
                 self.monitor_op = np.less
                 self.best = np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            if self.verbose > 0:
                print(f"ModelCheckpoint: Metric '{self.monitor}' not found in logs. Skipping checkpoint.")
            return

        filepath = self.filepath.format(epoch=epoch + 1, **logs)

        if self.save_best_only:
            if self.monitor_op(current, self.best):
                if self.verbose > 0:
                    print(f"\nEpoch {epoch+1}: {self.monitor} improved from {self.best:.4f} to {current:.4f}, saving model to {filepath}")
                self.best = current
                if self.save_weights_only:
                    torch.save(self.trainer.model.state_dict(), filepath)
                else:
                    torch.save(self.trainer.model, filepath)
            else:
                if self.verbose > 1:
                    print(f"\nEpoch {epoch+1}: {self.monitor} did not improve from {self.best:.4f}")
        else:
            if self.verbose > 0:
                print(f"\nEpoch {epoch+1}: saving model to {filepath}")
            if self.save_weights_only:
                 torch.save(self.trainer.model.state_dict(), filepath)
            else:
                 torch.save(self.trainer.model, filepath)


class ReduceLROnPlateau(Callback):
    """Reduce learning rate when a metric has stopped improving."""
    def __init__(self, monitor='val_loss', factor=0.1, patience=10, verbose=0, mode='auto',
                 min_delta=1e-4, cooldown=0, min_lr=0, epsilon=1e-8):
        super().__init__()
        self.monitor = monitor
        self.factor = factor
        self.patience = patience
        self.verbose = verbose
        self.min_delta = min_delta
        self.cooldown = cooldown
        self.min_lr = min_lr
        self.epsilon = epsilon
        self.wait = 0
        self.cooldown_counter = 0
        self.mode = mode

        if mode not in ['auto', 'min', 'max']:
            raise ValueError("mode must be one of 'auto', 'min', 'max'")
        self.mode = mode

        if self.mode == 'min':
            self.monitor_op = lambda a, b: np.less(a, b - self.min_delta)
            self.best = np.Inf
        elif self.mode == 'max':
            self.monitor_op = lambda a, b: np.greater(a, b + self.min_delta)
            self.best = -np.Inf
        else: # auto
             if 'acc' in self.monitor or 'f1' in self.monitor or 'auc' in self.monitor: # Example metrics where higher is better
                 self.monitor_op = lambda a, b: np.greater(a, b + self.min_delta)
                 self.best = -np.Inf
             else: # Assume loss, where lower is better
                 self.monitor_op = lambda a, b: np.less(a, b - self.min_delta)
                 self.best = np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            if self.verbose > 0:
                 print(f"ReduceLROnPlateau: Metric '{self.monitor}' not found in logs. Skipping LR reduction.")
            return

        if self.cooldown_counter > 0:
            self.cooldown_counter -= 1
            self.wait = 0

        if self.monitor_op(current, self.best):
            self.best = current
            self.wait = 0
        elif self.cooldown_counter == 0:
            self.wait += 1
            if self.wait >= self.patience:
                self._reduce_lr(epoch)
                self.cooldown_counter = self.cooldown
                self.wait = 0

    def _reduce_lr(self, epoch):
        old_lr = self.trainer.optimizer.param_groups[0]['lr']
        new_lr = max(old_lr * self.factor, self.min_lr)
        if old_lr - new_lr > self.epsilon:
            self.trainer.optimizer.param_groups[0]['lr'] = new_lr
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: ReduceLROnPlateau reducing learning rate from {old_lr:.10f} to {new_lr:.10f}.')

class CosineAnnealingWarmRestarts(Callback):
    """
    Implements cosine annealing with warm restarts for the learning rate.
    Equivalent to torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
    but implemented as a callback to integrate with the trainer class.
    """
    def __init__(self, T_0, T_mult=1, eta_min=0, last_epoch=-1, verbose=False):
        """
        Args:
            T_0 (int): Number of epochs for the first restart.
            T_mult (int): Factor increases T_i after each restart. Default: 1.
            eta_min (float): Minimum learning rate. Default: 0.
            last_epoch (int): The index of the last epoch. Default: -1.
            verbose (bool): If True, prints a message at each update. Default: False.
        """
        super().__init__()
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.last_epoch = last_epoch
        self.verbose = verbose
        self.T_curr = last_epoch + 1 # Current number of epochs since last restart
        self._last_lr = [] # Store the last learning rates per param group


    def set_optimizer(self, optimizer):
         super().set_optimizer(optimizer)
         # Store initial learning rates
         for param_group in self.optimizer.param_groups:
             self._last_lr.append(param_group['lr'])


    def on_epoch_begin(self, epoch, logs=None):
        # Calculate the learning rate for the current epoch
        lr_list = self.get_lr()

        # Update the learning rate in the optimizer
        for i, param_group in enumerate(self.optimizer.param_groups):
            param_group['lr'] = lr_list[i]

        self._last_lr = lr_list # Update last_lr
        self.T_curr += 1 # Increment current epoch count since restart


    def get_lr(self):
        """Calculates the learning rate for the current epoch."""
        if self.T_curr == 0:
            return [base_lr for base_lr in self._last_lr]
        elif (self.T_curr - 1 - self.T_0) % (2 * self.T_0) == 0:
             # Restart, calculate new T_0
             return [self.eta_min + (base_lr - self.eta_min) *
                     (1 + math.cos(math.pi / self.T_0)) / 2
                     for base_lr in self._last_lr]
        else:
             # Cosine annealing within the current cycle
             return [self.eta_min + (base_lr - self.eta_min) *
                     (1 + math.cos(math.pi * (self.T_curr % self.T_0) / self.T_0)) / 2
                     for base_lr in self._last_lr]


class ModelTrainer:
    """
    A comprehensive class for training and evaluating time series models
    with optional image feature integration for conflict prediction.
    Includes support for callbacks.
    """
    def __init__(self, df, img_dir, n_steps, forecast_horizon, target_col='hs_code',
                 val_test_split_ratio=0.3, batch_size=32, num_workers=2,
                 image_target_size=(256, 256),
                 model_type='combined_3d', # Options: 'cnn_2d_cls', 'cnn_2d_seq', 'cnn_3d_cls', 'cnn_3d_seq', 'ts_only', 'combined_2d', 'combined_3d'
                 model_params=None, callbacks=None):

        """
        Args:
            df (pd.DataFrame): The input DataFrame with time series data.
            img_dir (str): Directory with processed image files.
            n_steps (int): Number of time steps in the input sequence (X).
            forecast_horizon (int): The number of future time steps to forecast.
            target_col (str): The name of the target column in the DataFrame. Defaults to 'hs_code'.
            val_test_split_ratio (float): The combined proportion of data for validation and testing.
                                          Defaults to 0.3.
            batch_size (int): Batch size for DataLoaders. Defaults to 32.
            num_workers (int): Number of workers for DataLoaders. Defaults to 2.
            image_target_size (tuple): The target spatial size (height, width) for image
                                       resizing/padding. Defaults to (256, 256).
            model_type (str): The type of model to use. Defaults to 'combined_3d'.
                              See options in the comment above.
            model_params (dict, optional): Dictionary of parameters for the selected model.
                                           Required parameters depend on the model_type.
            callbacks (list, optional): A list of Callback instances to apply during training.
                                        Defaults to None.
        """
        super().__init__()
        self.df = df.copy()
        self.img_dir = img_dir
        self.n_steps = n_steps
        self.forecast_horizon = forecast_horizon
        self.target_col = target_col
        self.val_test_split_ratio = val_test_split_ratio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.image_target_size = image_target_size
        self.model_type = model_type
        self.model_params = model_params if model_params is not None else {}
        self.callbacks = callbacks if callbacks is not None else []

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        self.stop_training = False # Flag to signal early stopping

        self._validate_model_params()
        self._prepare_data()
        self._create_model()
        self._define_criterion()
        self._define_optimizer()
        self._set_callbacks()


    def _validate_model_params(self):
        """Validates that required model parameters are provided."""
        required_params = {
            'cnn_2d_cls': ['image_in_channels', 'num_classes'],
            'cnn_2d_seq': ['image_in_channels', 'num_features_extracted'],
            'cnn_3d_cls': ['image_in_channels', 'num_classes'],
            'cnn_3d_seq': ['image_in_channels', 'num_features_extracted'],
            'ts_only': ['numerical_feature_size', 'hidden_size', 'num_layers', 'num_classes'],
            'combined_2d': ['image_in_channels', 'num_features_extracted', 'numerical_feature_size', 'hidden_size', 'num_layers', 'num_classes'],
            'combined_3d': ['image_in_channels', 'num_features_extracted', 'numerical_feature_size', 'hidden_size', 'num_layers', 'num_classes'],
        }
        if self.model_type not in required_params:
             raise ValueError(f"Invalid model_type: {self.model_type}")

        for param in required_params[self.model_type]:
            if param not in self.model_params:
                # Allow numerical_feature_size to be None if not set, as it's calculated
                if param == 'numerical_feature_size' and self.model_params.get(param) is None:
                    continue
                raise ValueError(f"Missing required parameter '{param}' for model_type '{self.model_type}' in model_params.")

        # Add num_time_steps and forecast_horizon to model_params if not already there
        self.model_params['num_time_steps'] = self.n_steps
        if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq', 'ts_only']:
             self.model_params['forecast_horizon'] = self.forecast_horizon


    def _prepare_data(self):
        """Splits the DataFrame and creates Dataset and DataLoader instances."""
        train_df, val_df, test_df = split_dataframe_chronologically_by_country(self.df, self.val_test_split_ratio)

        # Determine numerical feature size based on the training DataFrame after splitting
        # Exclude 'iso3', 'year', 'month', 'hs_code', 'hs_name', 'processed_image_path'
        numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()

        exclude_cols = ['year']
        if self.target_col in numerical_cols:
             exclude_cols.append(self.target_col)

        self.numerical_feature_cols = [col for col in numerical_cols if col not in exclude_cols]
        self.numerical_feature_size = len(self.numerical_feature_cols)
        print(f"Using {self.numerical_feature_size} numerical features: {self.numerical_feature_cols}")

        # Update numerical_feature_size in model_params for relevant models
        if self.model_type in ['ts_only', 'combined_2d', 'combined_3d']:
             self.model_params['numerical_feature_size'] = self.numerical_feature_size

        # Initialize StandardScaler
        self.scaler = StandardScaler()

        # Fit scaler on the training data numerical features and transform
        train_df[self.numerical_feature_cols] = self.scaler.fit_transform(train_df[self.numerical_feature_cols])
        # Transform validation and test data numerical features
        val_df[self.numerical_feature_cols] = self.scaler.transform(val_df[self.numerical_feature_cols])
        test_df[self.numerical_feature_cols] = self.scaler.transform(test_df[self.numerical_feature_cols])


        self.train_dataset = ConflictDataset(train_df, self.img_dir, self.n_steps, self.forecast_horizon,
                                             target_col=self.target_col, target_size=self.image_target_size)
        self.val_dataset = ConflictDataset(val_df, self.img_dir, self.n_steps, self.forecast_horizon,
                                           target_col=self.target_col, target_size=self.image_target_size)
        self.test_dataset = ConflictDataset(test_df, self.img_dir, self.n_steps, self.forecast_horizon,
                                            target_col=self.target_col, target_size=self.image_target_size)

        # Create DataLoaders
        self.train_dataloader = DataLoader(self.train_dataset, batch_size=self.batch_size,
                                           shuffle=True, num_workers=self.num_workers)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=self.batch_size,
                                         shuffle=False, num_workers=self.num_workers)
        self.test_dataloader = DataLoader(self.test_dataset, batch_size=self.batch_size,
                                          shuffle=False, num_workers=self.num_workers)

        print(f"\nPrepared data:")
        print(f"Train dataset size: {len(self.train_dataset)}")
        print(f"Validation dataset size: {len(self.val_dataset)}")
        print(f"Test dataset size: {len(self.test_dataset)}")
        print(f"Number of batches in train dataloader: {len(self.train_dataloader)}")
        print(f"Number of batches in validation dataloader: {len(self.val_dataloader)}")
        print(f"Number of batches in test dataloader: {len(self.test_dataloader)}")


    def _create_model(self):
        """Instantiates the chosen model based on self.model_type."""
        if self.model_type == 'cnn_2d_cls':
            # Image-only 2D CNN for classification
            self.model = ImageSequenceFeatureExtractor2D(
                num_time_steps=self.n_steps,
                in_channels=self.model_params['image_in_channels'],
                combining_outputs=False, # Classification output
                forecast_horizon=self.forecast_horizon,
                num_classes=self.model_params['num_classes']
            )
        elif self.model_type == 'cnn_2d_seq':
             # Image-only 2D CNN outputting sequence features
             self.model = ImageSequenceFeatureExtractor2D(
                num_time_steps=self.n_steps,
                in_channels=self.model_params['image_in_channels'],
                num_features_extracted=self.model_params['num_features_extracted'],
                combining_outputs=True # Sequence feature output
            )
        elif self.model_type == 'cnn_3d_cls':
            # Image-only 3D CNN for classification
            self.model = ImageSequenceFeatureExtractor3D(
                num_time_steps=self.n_steps,
                in_channels=self.model_params['image_in_channels'],
                combining_outputs=False, # Classification output
                forecast_horizon=self.forecast_horizon,
                num_classes=self.model_params['num_classes']
            )
        elif self.model_type == 'cnn_3d_seq':
             # Image-only 3D CNN outputting sequence features
             self.model = ImageSequenceFeatureExtractor3D(
                num_time_steps=self.n_steps,
                in_channels=self.model_params['image_in_channels'],
                num_features_extracted=self.model_params['num_features_extracted'],
                combining_outputs=True # Sequence feature output
            )
        elif self.model_type == 'ts_only':
            # Time-series network using only numerical data
            self.model = TimeSeriesNetwork(
                combined_feature_size=self.numerical_feature_size, # Only numerical features
                hidden_size=self.model_params['hidden_size'],
                num_layers=self.model_params['num_layers'],
                output_mode='classification', # Classification output
                forecast_horizon=self.forecast_horizon,
                num_classes=self.model_params['num_classes'],
                dropout_prob=self.model_params.get('dropout_prob', 0.0)
            )
        elif self.model_type in ['combined_2d', 'combined_3d']:
            # Combined model with image features (from 2D or 3D extractor) and numerical data
            image_extractor_type = self.model_type.split('_')[1] # '2d' or '3d'
            self.model = CombinedModel(
                num_time_steps=self.n_steps,
                image_in_channels=self.model_params['image_in_channels'],
                num_features_extracted=self.model_params['num_features_extracted'],
                numerical_feature_size=self.numerical_feature_size, # Use calculated size
                hidden_size=self.model_params['hidden_size'],
                num_layers=self.model_params['num_layers'],
                forecast_horizon=self.forecast_horizon,
                num_classes=self.model_params['num_classes'],
                dropout_prob=self.model_params.get('dropout_prob', 0.0),
                image_feature_extractor_type=image_extractor_type.upper() # '2D' or '3D'
            )
        else:
            raise ValueError(f"Unsupported model_type: {self.model_type}")

        self.model.to(self.device)
        print(f"\nCreated model: {self.model_type}")
        # print(self.model) # Optional: print model architecture


    def _define_criterion(self):
        """Defines the loss function."""
        # Using CrossEntropyLoss for classification tasks
        # If the task were regression, you would use nn.MSELoss, etc.
        self.criterion = nn.CrossEntropyLoss()


    def _define_optimizer(self):
        """Defines the optimizer."""
        # Using Adam optimizer
        learning_rate = self.model_params.get('learning_rate', 0.001)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

    def _set_callbacks(self):
        """Sets up callbacks by assigning trainer, model, optimizer, and dataloaders."""
        for callback in self.callbacks:
            callback.set_trainer(self)
            callback.set_model(self.model)
            callback.set_optimizer(self.optimizer)
            callback.set_dataloader('train', self.train_dataloader)
            callback.set_dataloader('val', self.val_dataloader)
            callback.set_dataloader('test', self.test_dataloader) # Pass test dataloader if needed by callbacks


    def train_epoch(self, epoch):
        """Trains the model for one epoch."""
        self.model.train()
        running_loss = 0.0
        all_targets = []
        all_predictions = []
        start_time = time.time()

        logs = {}
        self.call_callbacks('on_epoch_begin', epoch, logs)

        for batch_idx, (numerical_sequences, image_sequences, targets, original_indices) in enumerate(self.train_dataloader):
            batch_logs = {}
            self.call_callbacks('on_batch_begin', batch_idx, batch_logs)

            # Move data to device
            numerical_sequences = numerical_sequences.to(self.device)
            image_sequences = image_sequences.to(self.device)
            targets = targets.to(self.device) # Shape [batch_size, forecast_horizon]

            # Zero the parameter gradients
            self.optimizer.zero_grad()

            # Forward pass
            # Model output shape for classification models: [batch_size, forecast_horizon, num_classes] (logits)
            # For sequence output models: [batch_size, num_steps, num_features_extracted]
            if self.model_type in ['ts_only', 'combined_2d', 'combined_3d', 'cnn_2d_cls', 'cnn_3d_cls']:
                 # Classification output
                 outputs = self.model(numerical_sequences, image_sequences)
                 # Calculate loss
                 # Need to reshape outputs and targets for CrossEntropyLoss
                 # Reshape outputs: [batch_size * forecast_horizon, num_classes]
                 # Reshape targets: [batch_size * forecast_horizon]
                 loss = self.criterion(outputs.view(-1, self.model_params['num_classes']), targets.view(-1))

                 # Store targets and predictions for accuracy calculation
                 all_targets.append(targets.cpu().numpy().flatten())
                 all_predictions.append(torch.argmax(outputs, dim=-1).cpu().numpy().flatten())

            elif self.model_type in ['cnn_2d_seq', 'cnn_3d_seq']:
                 # Sequence feature output (no direct loss calculation in this setup)
                 # This case is for models that output features to be used by another network
                 # For this trainer class, we'll focus on models with direct output (classification or regression)
                 # print(f"Warning: Model type '{self.model_type}' outputs sequence features. No loss calculated in train_epoch.")
                 loss = torch.tensor(0.0, device=self.device) # Dummy loss


            # Backward pass and optimize
            if loss.requires_grad: # Only backpropagate if loss is calculated
                 loss.backward()
                 self.optimizer.step()

            if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq']:
                running_loss += loss.item() * numerical_sequences.size(0)

            batch_logs['loss'] = loss.item()
            self.call_callbacks('on_batch_end', batch_idx, batch_logs)


        # Avoid division by zero if no loss was accumulated (e.g., for feature extractors)
        epoch_loss = running_loss / (len(self.train_dataloader.dataset) if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq'] else 1)
        epoch_duration = time.time() - start_time

        logs['train_loss'] = epoch_loss
        logs['epoch_duration'] = epoch_duration
        logs['lr'] = self.optimizer.param_groups[0]['lr'] # Log current learning rate

        # Calculate training accuracy if applicable
        if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq'] and len(all_targets) > 0:
            all_targets_combined = np.concatenate(all_targets)
            all_predictions_combined = np.concatenate(all_predictions)
            train_accuracy = accuracy_score(all_targets_combined, all_predictions_combined)
            logs['train_accuracy'] = train_accuracy
        else:
            train_accuracy = None


        # Evaluate on validation set after epoch
        val_loss, val_accuracy = self.evaluate_epoch(self.val_dataloader)
        logs['val_loss'] = val_loss
        logs['val_accuracy'] = val_accuracy


        self.call_callbacks('on_epoch_end', epoch, logs)

        return epoch_loss, train_accuracy, val_loss, val_accuracy # Return all metrics

    def evaluate_epoch(self, dataloader):
        """Evaluates the model on a given dataloader for one epoch."""
        self.model.eval()
        running_loss = 0.0
        all_targets = []
        all_predictions = []

        with torch.inference_mode():
            for batch_idx, (numerical_sequences, image_sequences, targets, original_indices) in enumerate(dataloader):
                # Move data to device
                numerical_sequences = numerical_sequences.to(self.device)
                image_sequences = image_sequences.to(self.device)
                targets = targets.to(self.device)

                # Forward pass
                if self.model_type in ['ts_only', 'combined_2d', 'combined_3d', 'cnn_2d_cls', 'cnn_3d_cls']:
                     outputs = self.model(numerical_sequences, image_sequences)
                     # Calculate loss
                     loss = self.criterion(outputs.view(-1, self.model_params['num_classes']), targets.view(-1))
                     running_loss += loss.item() * numerical_sequences.size(0)

                     # Store targets and predictions for accuracy calculation
                     all_targets.append(targets.cpu().numpy().flatten())
                     all_predictions.append(torch.argmax(outputs, dim=-1).cpu().numpy().flatten())

                elif self.model_type in ['cnn_2d_seq', 'cnn_3d_seq']:
                     # Feature extractors don't have a direct loss or accuracy for evaluation in this setup
                     pass # No calculation or accumulation


        # Avoid division by zero if no loss or accuracy was accumulated
        epoch_loss = running_loss / (len(dataloader.dataset) if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq'] else 1)

        if self.model_type not in ['cnn_2d_seq', 'cnn_3d_seq'] and len(all_targets) > 0:
            all_targets_combined = np.concatenate(all_targets)
            all_predictions_combined = np.concatenate(all_predictions)
            epoch_accuracy = accuracy_score(all_targets_combined, all_predictions_combined)
        else:
            epoch_accuracy = None # Not applicable for feature extractors

        return epoch_loss, epoch_accuracy


    def train(self, num_epochs):
        """Starts the training process."""
        print(f"\nStarting training for {num_epochs} epochs with model type: {self.model_type}")
        history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}

        self.call_callbacks('on_train_begin')
        self.stop_training = False # Reset stop flag

        for epoch in range(num_epochs):
            if self.stop_training: # Check if early stopping was triggered
                print(f"\nTraining stopped early at epoch {epoch+1}.")
                break

            train_loss, train_accuracy, val_loss, val_accuracy = self.train_epoch(epoch)

            train_accuracy_str = f"{train_accuracy:.4f}" if train_accuracy is not None else 'N/A'
            val_accuracy_str = f"{val_accuracy:.4f}" if val_accuracy is not None else 'N/A'

            print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy_str}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy_str}")

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            if train_accuracy is not None:
                history['train_accuracy'].append(train_accuracy)
            if val_accuracy is not None:
                 history['val_accuracy'].append(val_accuracy)


        self.call_callbacks('on_train_end')

        print("\nTraining finished.")
        return history


    def evaluate_test(self):
        """Evaluates the trained model on the test set."""
        print(f"\nEvaluating model on the test set...")
        test_loss, test_accuracy = self.evaluate_epoch(self.test_dataloader)
        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f if test_accuracy is not None else 'N/A'}")
        return test_loss, test_accuracy

    def predict(self, dataloader):
         """
         Generates predictions for a given dataloader.
         Returns raw model outputs (logits for classification models).
         """
         self.model.eval()
         predictions = []
         original_indices = []
         with torch.no_grad():
              for numerical_sequences, image_sequences, targets, batch_original_indices in dataloader:
                   numerical_sequences = numerical_sequences.to(self.device)
                   image_sequences = image_sequences.to(self.device)

                   # Model output shape for classification models: [batch_size, forecast_horizon, num_classes] (logits)
                   # For sequence output models: [batch_size, num_steps, num_features_extracted]
                   if self.model_type in ['ts_only', 'combined_2d', 'combined_3d', 'cnn_2d_cls', 'cnn_3d_cls']:
                        outputs = self.model(numerical_sequences, image_sequences)
                        predictions.append(outputs.cpu())
                   elif self.model_type in ['cnn_2d_seq', 'cnn_3d_seq']:
                        # Feature extractors output features, not direct predictions for the target variable
                        print(f"Warning: Model type '{self.model_type}' outputs sequence features. Prediction method not applicable for this model type.")
                        return None

                   original_indices.append(batch_original_indices)


         if predictions:
             # Concatenate predictions from all batches
             predictions = torch.cat(predictions, dim=0)
             original_indices = torch.cat(original_indices, dim=0)
             return predictions.numpy(), original_indices.numpy()
         else:
              return None, None

    def call_callbacks(self, method_name, *args, **kwargs):
        """Calls the specified method on all registered callbacks."""
        logs = kwargs.get('logs', {})
        for callback in self.callbacks:
            if hasattr(callback, method_name):
                getattr(callback, method_name)(*args, logs)

In [ ]:
# # Load your data
df = pd.read_csv('/kaggle/working/processed_df.csv')
img_dir = "/kaggle/input/agricultural-hotspot-fapar-rasters-conflict-info/fapar_patches_processed"

## Training the Model

### DEFINING GENERIC PARAMS

In [ ]:
hidden_size = 128
num_layers = 2
image_in_channels = 3
num_features_extracted = 64

### CONFIGURING MODELS

#### 3D CNN WITH TIMESERIES

##### N_TIME_STPES = 5, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_3d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_3d', # Choose your model type here
    model_params=model_params_combined_3d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

#### 2D CNN WITH TIMESERIES

##### N_TIME_STPES = 5, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_combined_2d = {
    'image_in_channels': image_in_channels,
    'num_features_extracted': num_features_extracted,
    'numerical_feature_size': None, # This will be calculated automatically by the trainer
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'num_classes': num_classes,
    'dropout_prob': 0.2,
    'learning_rate': 0.001
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='combined_2d', # Choose your model type here
    model_params=model_params_combined_2d
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

#### 3D CNN DIRECT PREDICTIONS

##### N_TIME_STPES = 5, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_3d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_3d_cls', # Choose your model type here
    model_params=model_params_3d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

#### 2D CNN WITH DIRECT CLASSIFICATION

##### N_TIME_STPES = 5, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_2d_cls = {
    'image_in_channels': image_in_channels,
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='cnn_2d_cls', # Choose your model type here
    model_params=model_params_2d_cls
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

#### TIMESERIES WITH DIRECT CLASSIFICATION

##### N_TIME_STPES = 5, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 5, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 4, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 4
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = **2**

In [ ]:
# # Define training parameters
n_steps = 3
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 3, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 5
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 3

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 3
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 2

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 2
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")

##### N_TIME_STPES = 6, FORECAST_HORIZON = 1

In [ ]:
# # Define training parameters
n_steps = 6
forecast_horizon = 1
target_col = 'hs_code' # The target variable
num_classes = df[target_col].nunique() # Get the number of unique classes from the data

# # Define model parameters based on the chosen model type
model_params_ts_only = {
    'hidden_size': hidden_size,
    'num_layers': num_layers
    'num_classes': num_classes
}

# # Create an instance of the ModelTrainer
trainer = ModelTrainer(
    df=df,
    img_dir=img_dir,
    n_steps=n_steps,
    forecast_horizon=forecast_horizon,
    target_col=target_col,
    val_test_split_ratio=0.3,
    batch_size=32,
    num_workers=2,
    image_target_size=(256, 256),
    model_type='ts_only', # Choose your model type here
    model_params=model_params_ts_only
)

In [ ]:
# # Train the model
num_epochs = 10
training_history = trainer.train(num_epochs)

# # Evaluate on the test set
test_loss = trainer.evaluate_test()

# # Generate predictions on the test set (optional)
predictions, original_indices = trainer.predict(trainer.test_dataloader)
if predictions is not None:
    print(f"\nPredictions shape on test set: {predictions.shape}")
    # Example: Get predicted class for the first step of the forecast horizon
    predicted_classes_step1 = np.argmax(predictions[:, 0, :], axis=-1)
    print(f"Predicted classes for the first forecast step (first 5 samples): {predicted_classes_step1[:5]}")